# PDF からテキストを抽出する (PyMuPDF)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_data_io/pdf_extract.ipynb)

NLP クックブック 01-04 ／ データの収集と入出力

論文や報告書の PDF から本文テキストを取り出したい。ページごと、段組みを考慮して取りたい。

In [1]:
!pip install -q pymupdf

In [2]:
import urllib.request
import fitz  # PyMuPDF

urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "paper.pdf")   # Attention Is All You Need
doc = fitz.open("paper.pdf")
print(len(doc), "ページ", doc.metadata.get("title"))

page = doc[0]
print(page.get_text()[:400])

15 ページ 
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G


In [3]:
import re

# ブロック (段落) 単位で取り出すと、位置情報が得られ、読み順の制御やヘッダ・フッタの除去ができる
blocks = doc[1].get_text("blocks")     # (x0, y0, x1, y1, テキスト, ブロック番号, 種類)
for x0, y0, x1, y1, text, *_ in blocks[:4]:
    print(f"({x0:5.0f}, {y0:5.0f})  {text[:70]!r}")

def clean(text):
    text = re.sub(r"-\n(\w)", r"\1", text)      # 行末のハイフネーションをつなぐ
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)   # 段落内の改行を空白に (日本語の場合は空文字に)
    return re.sub(r"[ \t]+", " ", text).strip()

full = "\n\n".join(clean(b[4]) for p in doc for b in p.get_text("blocks", sort=True) if b[6] == 0)
print(len(full), "文字")
print(full[1500:1900])

(  108,    73)  '1\nIntroduction\n'
(  108,    99)  'Recurrent neural networks, long short-term memory [13] and gated recur'
(  108,   159)  'Recurrent models typically factor computation along the symbol positio'
(  108,   252)  'Attention mechanisms have become an integral part of compelling sequen'
40011 文字
 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data.

∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started


- PDF は「文字と座標の集まり」で、段落や読み順の情報を持たない。2 段組み、表、数式、脚注は崩れやすい。
- スキャンされた PDF（画像）からは抽出できない。OCR が必要（`page.get_textpage_ocr()` や、専用の OCR / 文書解析モデル）。画像の OCR は、画像と言語の章の「画像の中の日本語を読み取り (OCR)、文字誤り率で比べる」と「PDF のページを画像にして視覚言語モデルで読む（表とグラフ）」を参照。
- 論文の構造（章、参考文献）まで欲しいなら GROBID、レイアウトを保った Markdown が欲しいなら `pymupdf4llm` や Docling などがある。